<a href="https://colab.research.google.com/github/duy30052005/Satellite-Image-Segmentation/blob/main/demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# Cài đặt và ép nâng cấp (-U) các thư viện lõi để chống xung đột
!pip install -q -U geemap xyzservices python-box streamlit streamlit_folium pyngrok

!pip install segmentation-models-pytorch

In [8]:
%%writefile app.py
import streamlit as st
import folium
from streamlit_folium import st_folium
import os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
import torch.nn.functional as F
import math
import requests
from PIL import Image
from io import BytesIO
import plotly.express as px

# ==========================================
# CẤU HÌNH TRANG & BIẾN TOÀN CỤC
# ==========================================
st.set_page_config(layout="wide", page_title="Hệ thống GCS & AI Giám sát DACN1", page_icon="🚁")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if 'drone_pos' not in st.session_state: st.session_state['drone_pos'] = None
if 'map_center' not in st.session_state: st.session_state['map_center'] = [16.047, 108.206]
if 'roi_coords' not in st.session_state: st.session_state['roi_coords'] = None
if 'captured_image' not in st.session_state: st.session_state['captured_image'] = None

# ==========================================
# SIDEBAR: CẤU HÌNH KỸ THUẬT & BAY
# ==========================================
with st.sidebar:
    st.title("⚙️ Cấu hình Hệ thống")
    st.markdown("---")
    st.subheader("🚁 Thông số Cảm biến Drone")
    # Mở rộng trần bay lên 2000m để quét diện tích lớn
    altitude = st.slider("Độ cao bay (Altitude - mét):", min_value=10, max_value=2000, value=150, step=10)
    fov = st.slider("Góc nhìn (FOV - độ):", min_value=60, max_value=90, value=84, step=1)

    st.markdown("---")
    st.subheader("🧠 Động cơ AI (U-Net)")
    weight_path = st.text_input("Đường dẫn file trọng số (.pth):", value="/content/drive/MyDrive/Colab Notebooks/DACN1/unet_resnet50_V4_Advanced_HDC.pth")

# ==========================================
# GIAO DIỆN CHÍNH
# ==========================================
st.title("🛰️ Trạm Quan Trắc & Phân Tích Hiện Trạng Bằng Flycam (DACN1)")

# ---------------------------------------------------------
# BƯỚC 1: ĐIỀU KHIỂN FLYCAM LẤY DỮ LIỆU HIỆN TẠI
# ---------------------------------------------------------
st.header("📍 Bước 1: Thu thập Dữ liệu Hiện tại (Live Capture)")

fov_rad = math.radians(fov)
ground_width_m = 2 * altitude * math.tan(fov_rad / 2)
gsd_cm = (ground_width_m / 512.0) * 100.0

def create_drone_map():
    m = folium.Map(location=st.session_state['map_center'], zoom_start=15)
    folium.TileLayer(
        tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
        attr='Esri World', name='Esri Satellite', overlay=False, control=True
    ).add_to(m)

    if st.session_state['drone_pos']:
        lat = st.session_state['drone_pos']['lat']
        lng = st.session_state['drone_pos']['lng']
        folium.Marker([lat, lng], tooltip="Tâm Drone", icon=folium.Icon(color="red", icon="info-sign")).add_to(m)

        lat_degree_m = 111320.0
        lon_degree_m = 111320.0 * math.cos(math.radians(lat))
        delta_lat = (ground_width_m / 2) / lat_degree_m
        delta_lon = (ground_width_m / 2) / lon_degree_m

        min_lat, max_lat = lat - delta_lat, lat + delta_lat
        min_lon, max_lon = lng - delta_lon, lng + delta_lon

        st.session_state['roi_coords'] = {
            'min_lon': min_lon, 'max_lon': max_lon,
            'min_lat': min_lat, 'max_lat': max_lat,
            'ground_width_m': ground_width_m
        }

        folium.Rectangle(
            bounds=[[min_lat, min_lon], [max_lat, max_lon]],
            color='#ff0000', weight=3, fill=True, fill_opacity=0.15,
            tooltip=f"Vùng chụp: {ground_width_m:.1f}m"
        ).add_to(m)
    return m

output = st_folium(create_drone_map(), height=450, use_container_width=True)

if output and output.get("last_clicked"):
    clicked_lat = output["last_clicked"]["lat"]
    clicked_lng = output["last_clicked"]["lng"]
    current_pos = st.session_state.get('drone_pos')
    if current_pos is None or (clicked_lat != current_pos['lat'] or clicked_lng != current_pos['lng']):
        st.session_state['drone_pos'] = {"lat": clicked_lat, "lng": clicked_lng}
        st.session_state['map_center'] = [clicked_lat, clicked_lng]
        st.session_state['captured_image'] = None
        st.rerun()

if st.session_state['drone_pos']:
    col_info1, col_info2, col_info3 = st.columns(3)
    col_info1.metric("Kích thước vùng quét", f"{ground_width_m:.1f} x {ground_width_m:.1f} m")
    col_info2.metric(f"Độ sắc nét (GSD)", f"{gsd_cm:.1f} cm/px", "Đạt chuẩn 🟢" if gsd_cm < 30 else "Ảnh mờ 🔴")

    with col_info3:
        st.write("")
        if st.button("📸 Lấy ảnh Flycam Hiện tại", type="primary", use_container_width=True):
            with st.spinner("Đang truyền dữ liệu quang học..."):
                c = st.session_state['roi_coords']
                url_export = "https://services.arcgisonline.com/arcgis/rest/services/World_Imagery/MapServer/export"
                params = {
                    "bbox": f"{c['min_lon']},{c['min_lat']},{c['max_lon']},{c['max_lat']}",
                    "bboxSR": "4326", "imageSR": "4326", "size": "512,512", "format": "png", "f": "image"
                }
                try:
                    response = requests.get(url_export, params=params)
                    if response.status_code == 200:
                        img = Image.open(BytesIO(response.content)).convert('RGB')
                        st.session_state['captured_image'] = np.array(img)
                        st.rerun()
                except Exception as e:
                    st.error(f"Lỗi API: {e}")

st.divider()

# ---------------------------------------------------------
# BƯỚC 2: AI PHÂN TÍCH DIỆN TÍCH TỨC THỜI (6 LỚP ĐỘC LẬP)
# ---------------------------------------------------------
@st.cache_resource
def load_ai_model(path):
    # Khởi tạo model với đúng 6 classes
    model = smp.Unet(encoder_name="resnet50", encoder_weights=None, in_channels=3, classes=6).to(DEVICE)
    model.load_state_dict(torch.load(path, map_location=DEVICE))
    model.eval()
    return model

if st.session_state['captured_image'] is not None:
    st.header("🧠 Bước 2: AI Phân tích Hiện trạng (6 Phân lớp)")

    img_live = st.session_state['captured_image']

    if st.button("🚀 Chạy Phân Đoạn Bề Mặt", type="primary", use_container_width=True):
        if not os.path.exists(weight_path):
            st.error(f"❌ Không tìm thấy file trọng số tại: {weight_path}")
        else:
            with st.spinner("AI đang bóc tách không gian (Giữ nguyên 6 nhãn gốc)..."):
                model = load_ai_model(weight_path)
                transform = A.Compose([A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)), ToTensorV2()])

                input_tensor = transform(image=img_live)["image"].unsqueeze(0).to(DEVICE)
                with torch.no_grad():
                    # Kỹ thuật TTA (Test Time Augmentation)
                    p_normal = F.softmax(model(input_tensor), dim=1)
                    p_hf = torch.flip(F.softmax(model(torch.flip(input_tensor, dims=[3])), dim=1), dims=[3])
                    p_vf = torch.flip(F.softmax(model(torch.flip(input_tensor, dims=[2])), dim=1), dims=[2])

                    # Trả ra output 6 lớp nguyên bản
                    mask_live = torch.argmax((p_normal + p_hf + p_vf)/3.0, dim=1).cpu().squeeze().numpy()

                # --- TRỰC QUAN HÓA KẾT QUẢ (6 LỚP) ---
                class_names = [
                    'Urban (Đô thị)',
                    'Agriculture (Nông nghiệp)',
                    'Rangeland (Đồng cỏ)',
                    'Forest (Rừng)',
                    'Water (Nước)',
                    'Barren (Đất trống)'
                ]

                # Bảng màu tương ứng chuyên biệt
                custom_colors = [
                    '#d62728', # Đỏ (Đô thị)
                    '#bcbd22', # Hồng (Nông nghiệp)
                    '#9467bd', # Cam (Đồng cỏ)
                    '#2ca02c', # Xanh lục đậm (Rừng)
                    '#1f77b4', # Xanh dương (Nước)
                    '#8c564b'  # Xám (Đất trống)
                ]
                cmap_custom = ListedColormap(custom_colors)

                fig, axes = plt.subplots(1, 2, figsize=(14, 6))
                axes[0].imshow(img_live); axes[0].set_title("Camera Flycam Thực Tế", fontsize=12); axes[0].axis('off')
                axes[1].imshow(mask_live, cmap=cmap_custom, vmin=0, vmax=5); axes[1].set_title("AI Phân Đoạn (6 Nhãn)", fontsize=12); axes[1].axis('off')

                # Chú thích (Legend)
                patches = [mpatches.Patch(color=custom_colors[i], label=class_names[i]) for i in range(6)]
                fig.legend(handles=patches, loc='lower center', bbox_to_anchor=(0.5, -0.05), ncol=3, fontsize=11)
                plt.tight_layout()
                st.pyplot(fig)

                st.divider()

                # --- THỐNG KÊ VẬT LÝ ---
                roi_area_ha = (ground_width_m ** 2) / 10000.0
                pixel_to_ha = roi_area_ha / 262144.0 # 512x512

                counts_live = [np.sum(mask_live == i) for i in range(6)]
                areas_live_ha = [c * pixel_to_ha for c in counts_live]
                areas_live_m2 = [a * 10000 for a in areas_live_ha]

                df_stats = pd.DataFrame({
                    'Hạng mục địa hình': class_names,
                    'Diện tích (m²)': [round(x, 1) for x in areas_live_m2],
                    'Diện tích (Hecta)': [round(x, 4) for x in areas_live_ha],
                    'Tỷ lệ bao phủ (%)': [round((c / 262144.0) * 100, 2) for c in counts_live]
                })

                st.subheader("📊 Báo cáo Thống kê Không gian Chuyên sâu")

                # --- VẼ BIỂU ĐỒ TƯƠNG TÁC BẰNG PLOTLY ---
                col_chart1, col_chart2 = st.columns(2)

                with col_chart1:
                    # Biểu đồ tròn (Pie Chart) thể hiện Tỷ lệ bao phủ
                    fig_pie = px.pie(
                        df_stats,
                        values='Tỷ lệ bao phủ (%)',
                        names='Hạng mục địa hình',
                        title='Tỷ trọng Phân bổ Bề mặt (%)',
                        color='Hạng mục địa hình',
                        color_discrete_sequence=custom_colors
                    )
                    fig_pie.update_traces(textposition='inside', textinfo='percent')
                    fig_pie.update_layout(margin=dict(t=40, b=0, l=0, r=0))
                    st.plotly_chart(fig_pie, use_container_width=True)

                with col_chart2:
                    # Biểu đồ cột (Bar Chart) thể hiện Diện tích vật lý
                    fig_bar = px.bar(
                        df_stats,
                        x='Hạng mục địa hình',
                        y='Diện tích (Hecta)',
                        title='Diện tích Vật lý Thực tế (Hecta)',
                        color='Hạng mục địa hình',
                        color_discrete_sequence=custom_colors,
                        text_auto='.2f'
                    )
                    fig_bar.update_layout(
                        margin=dict(t=40, b=0, l=0, r=0),
                        xaxis_title="",
                        yaxis_title="Hecta (Ha)",
                        showlegend=False
                    )
                    st.plotly_chart(fig_bar, use_container_width=True)

                # --- BẢNG DỮ LIỆU CHI TIẾT ---
                st.markdown("### 📋 Bảng Số Liệu Chi Tiết")
                st.dataframe(df_stats, use_container_width=True)

Overwriting app.py


In [7]:
from pyngrok import ngrok
import time

# 1. Tắt các tiến trình ngrok cũ (tránh lỗi trùng cổng khi chạy lại ô này)
ngrok.kill()

# 2. Nhập Token Ngrok của Khang
!ngrok config add-authtoken 3C3RZdHrDjn6IoAnksQTZ5Bfn2r_3Kz65G1jJGCcZhjKPC3AW

# 3. Chạy Streamlit ngầm
get_ipython().system_raw('streamlit run app.py &')
time.sleep(3) # Chờ Streamlit khởi động xong

# 4. Mở đường hầm Ngrok
pub_url = ngrok.connect(8501)
print("🚀 MÔI TRƯỜNG STREAMLIT CỦA BẠN ĐÃ ONLINE TẠI:")
print(pub_url)

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
🚀 MÔI TRƯỜNG STREAMLIT CỦA BẠN ĐÃ ONLINE TẠI:
NgrokTunnel: "https://tanna-heatful-antonietta.ngrok-free.dev" -> "http://localhost:8501"
